# Lab 0C Assignment: Build a Small Agent Card

Use this notebook after you finish `02_agent_walkthrough.ipynb`.

In this assignment, you will:
- run a plain baseline prompt on the same mini case packet
- edit a small agent card
- rerun the same model with your bounded agent design
- compare the results

The goal is not to build a perfect agent. The goal is to practice turning a vague model task into a more inspectable workflow.


In [ ]:
import csv
import json
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values
from openai import OpenAI

cwd = Path.cwd().resolve()
repo_root = cwd.parent
data_dir = cwd / 'data'

if not (repo_root / '.env.example').exists() or not (repo_root / 'src').exists():
    raise FileNotFoundError(
        'Open this notebook from its lab folder so the repo root is one level up.'
    )

config = dotenv_values(repo_root / '.env')
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

client = OpenAI(base_url=ollama_base_url, api_key='ollama')


def read_markdown(path: Path) -> str:
    return path.read_text(encoding='utf-8').strip()


def read_json_file(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def read_csv_rows(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))


case_brief = read_markdown(data_dir / 'case_brief.md')
artifact_manifest = read_json_file(data_dir / 'artifact_manifest.json')
device_events = read_csv_rows(data_dir / 'triage_events.csv')

case_packet = '\n\n'.join([
    'CASE BRIEF\n' + case_brief,
    'ARTIFACT MANIFEST\n' + json.dumps(artifact_manifest, indent=2),
    'DEVICE EVENTS\n' + json.dumps(device_events, indent=2),
])

toolbox = [
    'read_case_brief',
    'read_manifest',
    'read_device_events',
]


# Teaching note:
# `ask_model(...)` is the step that sends one prompt to the configured model.
# It also captures timing and raw text so students can compare different workflow styles.
def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
    }


# Teaching note:
# Models sometimes wrap JSON in code fences or add extra text around it.
# `clean_json_text(...)` trims that noise so the notebook can parse the response more reliably.
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned


print('Repo root:', repo_root)
print('Default model from .env:', default_model)
print('Case ID:', artifact_manifest['case_id'])
print('Approved tools:', ', '.join(toolbox))


## Step 1: Run a Plain Baseline Prompt

Start with a plain question so you have something to compare against after you design your agent card.


In [ ]:
baseline_prompt = f"""
Review this mini device-activity case packet and explain what happened and what should be checked next.

Case packet:
{case_packet}
""".strip()

baseline_response = ask_model(default_model, baseline_prompt)

print('Baseline prompt preview:\n')
print(baseline_prompt[:1800])
print('\nBaseline response:')
print('Model:', baseline_response['model'])
print('Time (seconds):', baseline_response['seconds'])
print('-' * 80)
print(baseline_response['raw_text'])


## Step 2: Edit the Agent Card

Edit the values below before running the next cell.

Keep the task limited. A good Lab 0 agent should have:
- one clear role
- one clear goal
- approved tools
- short memory notes
- a human-review boundary
- a stop condition


In [ ]:
student_agent_card = {
    'agent_name': 'Edit Me: Device Review Agent',
    'role': 'Edit this role so the agent has one limited device-review job.',
    'goal': 'Edit this goal so the agent knows what success looks like.',
    'approved_tools': toolbox,
    'memory': [
        'Edit this memory note.',
        'Edit this memory note.',
    ],
    'required_output_keys': [
        'observations',
        'unknowns',
        'next_step',
        'needs_human_review',
    ],
    'stop_condition': 'Stop after returning the required JSON object.',
    'human_review_rule': 'Edit this rule to say when a human must review the result.',
}

print(json.dumps(student_agent_card, indent=2))


## Step 3: Run Your Agent Design

The next cell turns your edited agent card into a prompt, runs the same model again, and checks whether your agent card looks customized.


In [ ]:
# Teaching note:
# `build_agent_prompt(...)` turns the agent card fields into one bounded prompt.
# This is where role, memory, rules, and stop condition become agent workflow structure.
def build_agent_prompt(agent: dict, packet: str) -> str:
    tool_lines = '\n'.join(f'- {name}' for name in agent['approved_tools'])
    memory_lines = '\n'.join(f'- {note}' for note in agent['memory'])
    required_keys = ', '.join(agent['required_output_keys'])
    return f"""
You are {agent['agent_name']}.

Role:
{agent['role']}

Goal:
{agent['goal']}

Approved tools:
{tool_lines}

Memory:
{memory_lines}

Rules:
- Use only the information from the provided case packet.
- Do not invent missing events or conclusions.
- Keep observations separate from unknowns.
- Apply this human review rule: {agent['human_review_rule']}
- Return valid JSON only.
- Use exactly these keys: {required_keys}.

Stop condition:
{agent['stop_condition']}

Case packet:
{packet}
""".strip()


remaining_placeholders = []
for key in ['agent_name', 'role', 'goal', 'human_review_rule']:
    if 'Edit this' in student_agent_card[key] or 'Edit Me' in student_agent_card[key]:
        remaining_placeholders.append(key)
if any('Edit this' in note for note in student_agent_card['memory']):
    remaining_placeholders.append('memory')

if remaining_placeholders:
    print('Reminder: customize these fields before trusting the result:', remaining_placeholders)

student_prompt = build_agent_prompt(student_agent_card, case_packet)
student_response = ask_model(default_model, student_prompt)

print('\nStudent agent prompt preview:\n')
print(student_prompt[:2200])
print('\nStudent agent response:')
print('Model:', student_response['model'])
print('Time (seconds):', student_response['seconds'])
print('-' * 80)
print(student_response['raw_text'])

student_json = None
try:
    student_json = json.loads(clean_json_text(student_response['raw_text']))
except Exception as exc:
    print('\nStudent agent JSON parse error:', exc)

if student_json is not None:
    print('\nParsed student agent JSON:')
    print(json.dumps(student_json, indent=2))

design_check = {
    'role_customized': 'Edit this' not in student_agent_card['role'],
    'goal_customized': 'Edit this' not in student_agent_card['goal'],
    'memory_customized': all('Edit this' not in note for note in student_agent_card['memory']),
    'human_review_rule_customized': 'Edit this' not in student_agent_card['human_review_rule'],
    'uses_tool_list': bool(student_agent_card['approved_tools']),
    'has_stop_condition': bool(student_agent_card['stop_condition']),
}

print('\nAgent-card design check:')
print(json.dumps(design_check, indent=2))


## Reflection Questions

Replace this text with short answers to the questions below.

1. Which part of your agent card changed the behavior of the model the most?
2. What decision did you leave for human review rather than the agent?
3. Did your agent output feel more inspectable than the baseline response? Why or why not?
4. In one sentence, explain what makes your final design an agent workflow instead of only a chatbot prompt.

## Submission

Save the notebook with:
- the baseline prompt and response
- your edited agent card
- your student agent response
- the design-check output
- your short reflection answers
